# 03. nn.Module Basics

## 학습 목표

1. `nn.Module`이 PyTorch 모델의 기본 클래스인 이유를 이해한다.
2. `__init__()`과 `forward()`의 역할을 구분한다.
3. `nn.Linear`, `nn.ReLU`, `nn.Sequential`을 사용한다.
4. 모델의 Parameter와 Module 구조를 확인한다.
5. `train()`과 `eval()` 모드를 구분한다.
6. 간단한 신경망 클래스를 직접 작성한다.


In [2]:
import torch
import torch.nn as nn

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cpu


---

# 1. `nn.Module`이란?

PyTorch에서 신경망 모델과 Layer는 대부분 `nn.Module`을 상속한다.

`nn.Module`은 다음 기능을 제공한다.

- Layer와 Parameter 자동 등록
- CPU/GPU 이동
- 학습/평가 모드 전환
- 모델 저장과 불러오기
- 모델 구조 출력


기본 구조:

```python
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Layer 정의

    def forward(self, x):
        # 입력의 연산 흐름 정의
        return x
```


---

# 2. `__init__()`과 `forward()`

- `__init__()`: 모델이 사용할 Layer를 생성한다.
- `forward()`: 입력 Tensor가 Layer를 통과하는 순서를 정의한다.


In [3]:
class SimpleLinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(in_features=3, out_features=1)

    def forward(self, x):
        return self.linear(x)


model = SimpleLinearModel()

print(model)

SimpleLinearModel(
  (linear): Linear(in_features=3, out_features=1, bias=True)
)


`nn.Linear(3, 1)`은 다음 연산을 수행한다.

$$
\mathbf{y}=\mathbf{x}\mathbf{W}^{T}+\mathbf{b}
$$


In [4]:
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])

output = model(x)

print("입력 shape:", x.shape)
print("출력 shape:", output.shape)
print("출력:\n", output)

입력 shape: torch.Size([2, 3])
출력 shape: torch.Size([2, 1])
출력:
 tensor([[1.4002],
        [2.0638]], grad_fn=<AddmmBackward0>)


shape 변화:

```text
(batch_size, 3)
→ Linear(3, 1)
→ (batch_size, 1)
```


---

# 3. 왜 `model.forward(x)` 대신 `model(x)`를 사용할까?

`model(x)`는 내부적으로 `forward()`를 호출하면서 Hook 등 `nn.Module`의 부가 기능도 처리한다.

따라서 일반적으로 다음처럼 호출한다.

```python
output = model(x)
```


In [5]:
output_1 = model(x)
output_2 = model.forward(x)

print(torch.allclose(output_1, output_2))

True


---

# 4. `nn.Linear`

`nn.Linear(in_features, out_features)`의 Parameter:

- Weight shape: `(out_features, in_features)` (당연하다. 선형대수에서 행렬 곱 생각하면 됨.)
- Bias shape: `(out_features,)` (당연하다. shape가 맞아야 더할 수 있다.)


In [6]:
layer = nn.Linear(4, 2)

print("weight shape:", layer.weight.shape)
print("bias shape:", layer.bias.shape)

print("\nweight:\n", layer.weight)
print("\nbias:\n", layer.bias)

weight shape: torch.Size([2, 4])
bias shape: torch.Size([2])

weight:
 Parameter containing:
tensor([[ 0.1713, -0.0192, -0.0926,  0.4179],
        [-0.2813, -0.0564, -0.1241, -0.3270]], requires_grad=True)

bias:
 Parameter containing:
tensor([ 0.2384, -0.3391], requires_grad=True)


In [7]:
x = torch.randn(5, 4)
y = layer(x)

print("입력 shape:", x.shape)
print("출력 shape:", y.shape)

입력 shape: torch.Size([5, 4])
출력 shape: torch.Size([5, 2])


마지막 차원의 크기 4가 2로 변한다.

```text
(5, 4) → (5, 2)
```


---

# 5. 활성화 함수

선형 Layer만 여러 개 연결하면 전체 모델도 하나의 선형 변환으로 합칠 수 있다.

복잡한 비선형 관계를 학습하기 위해 활성화 함수를 사용한다.


In [8]:
relu = nn.ReLU()

x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])

print(relu(x))

tensor([0., 0., 0., 1., 2.])


ReLU:

$$
\operatorname{ReLU}(x)=\max(0,x)
$$


---

# 6. 여러 Layer를 가진 모델


In [9]:
class TwoLayerNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


model = TwoLayerNetwork(input_dim=4, hidden_dim=8, output_dim=3)

print(model)

TwoLayerNetwork(
  (fc1): Linear(in_features=4, out_features=8, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=8, out_features=3, bias=True)
)


In [10]:
x = torch.randn(5, 4)
output = model(x)

print("입력 shape:", x.shape)
print("출력 shape:", output.shape)

입력 shape: torch.Size([5, 4])
출력 shape: torch.Size([5, 3])


shape 변화:

```text
(5, 4)
→ Linear(4, 8)
→ (5, 8)
→ ReLU
→ (5, 8)
→ Linear(8, 3)
→ (5, 3)
```


---

# 7. `nn.Sequential`

Layer가 순서대로 실행되는 단순한 구조는 `nn.Sequential`로 작성할 수 있다.


In [11]:
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))

print(model)

x = torch.randn(5, 4)
output = model(x)

print("출력 shape:", output.shape)

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=3, bias=True)
)
출력 shape: torch.Size([5, 3])


## 클래스 방식과 `Sequential`

### 직접 클래스 작성

- 복잡한 Forward 흐름 표현 가능
- Skip Connection이나 분기 구현 가능
- 중간값을 세밀하게 처리 가능

### `nn.Sequential`

- 단순한 순차 구조를 짧게 표현
- 읽기 쉽고 편리함
- 복잡한 분기에는 제한적


---

# 8. Parameter 확인하기


In [12]:
model = TwoLayerNetwork(4, 8, 3)

for index, parameter in enumerate(model.parameters()):
    print(
        f"parameter {index}: "
        f"shape={tuple(parameter.shape)}, "
        f"requires_grad={parameter.requires_grad}"
    )

parameter 0: shape=(8, 4), requires_grad=True
parameter 1: shape=(8,), requires_grad=True
parameter 2: shape=(3, 8), requires_grad=True
parameter 3: shape=(3,), requires_grad=True


In [13]:
for name, parameter in model.named_parameters():
    print(name, tuple(parameter.shape))

fc1.weight (8, 4)
fc1.bias (8,)
fc2.weight (3, 8)
fc2.bias (3,)


이 모델의 Parameter:

- `fc1.weight`
- `fc1.bias`
- `fc2.weight`
- `fc2.bias`


## Parameter 개수 계산

`numel()`은 number of element이다.


In [14]:
total_parameters = sum(parameter.numel() for parameter in model.parameters())

trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)

print("전체 Parameter 수:", total_parameters)
print("학습 가능한 Parameter 수:", trainable_parameters)

전체 Parameter 수: 67
학습 가능한 Parameter 수: 67


계산:

$$
8\times4+8=40
$$

$$
3\times8+3=27
$$

$$
40+27=67
$$


---

# 9. `children()`과 `modules()`

- `children()`: 바로 아래의 자식 Module
- `modules()`: 자기 자신을 포함한 모든 하위 Module


In [15]:
model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Sequential(nn.Linear(8, 4), nn.ReLU()),
    nn.Linear(4, 2),
)

print("children():")
for module in model.children():
    print(module)

print("\nmodules():")
for module in model.modules():
    print(module)

children():
Linear(in_features=4, out_features=8, bias=True)
ReLU()
Sequential(
  (0): Linear(in_features=8, out_features=4, bias=True)
  (1): ReLU()
)
Linear(in_features=4, out_features=2, bias=True)

modules():
Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Sequential(
    (0): Linear(in_features=8, out_features=4, bias=True)
    (1): ReLU()
  )
  (3): Linear(in_features=4, out_features=2, bias=True)
)
Linear(in_features=4, out_features=8, bias=True)
ReLU()
Sequential(
  (0): Linear(in_features=8, out_features=4, bias=True)
  (1): ReLU()
)
Linear(in_features=8, out_features=4, bias=True)
ReLU()
Linear(in_features=4, out_features=2, bias=True)


---

# 10. `state_dict()`

`state_dict()`는 모델의 Weight와 Bias를 이름과 함께 저장한 Dictionary이다.


In [16]:
model = TwoLayerNetwork(4, 8, 3)

state = model.state_dict()

for key, value in state.items():
    print(key, tuple(value.shape))

fc1.weight (8, 4)
fc1.bias (8,)
fc2.weight (3, 8)
fc2.bias (3,)


일반적인 저장과 불러오기:

```python
torch.save(model.state_dict(), "model.pt")

model.load_state_dict(
    torch.load("model.pt", weights_only=True)
)
```


---

# 11. CPU와 GPU 이동

모델과 입력 Tensor는 같은 Device에 있어야 한다.


In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TwoLayerNetwork(4, 8, 3).to(device)
x = torch.randn(5, 4).to(device)

output = model(x)

print("device:", device)
print("모델 device:", next(model.parameters()).device)
print("입력 device:", x.device)
print("출력 device:", output.device)

device: cpu
모델 device: cpu
입력 device: cpu
출력 device: cpu


---

# 12. `train()`과 `eval()`

- `model.train()`: 학습 모드
- `model.eval()`: 평가 모드

Dropout과 Batch Normalization처럼 모드에 따라 동작이 달라지는 Layer에 중요하다.


In [18]:
torch.manual_seed(42)

dropout_model = nn.Sequential(nn.Linear(4, 4), nn.ReLU(), nn.Dropout(p=0.5))

x = torch.ones(1, 4)

dropout_model.train()
train_output_1 = dropout_model(x)
train_output_2 = dropout_model(x)

dropout_model.eval()
eval_output_1 = dropout_model(x)
eval_output_2 = dropout_model(x)

print("학습 출력 1:", train_output_1)
print("학습 출력 2:", train_output_2)
print("\n평가 출력 1:", eval_output_1)
print("평가 출력 2:", eval_output_2)

학습 출력 1: tensor([[3.0498, 0.2309, 0.0000, 0.0000]], grad_fn=<MulBackward0>)
학습 출력 2: tensor([[3.0498, 0.0000, 0.7374, 0.0000]], grad_fn=<MulBackward0>)

평가 출력 1: tensor([[1.5249, 0.1155, 0.3687, 0.7351]], grad_fn=<ReluBackward0>)
평가 출력 2: tensor([[1.5249, 0.1155, 0.3687, 0.7351]], grad_fn=<ReluBackward0>)


평가 시 일반적인 패턴:

```python
model.eval()

with torch.no_grad():
    prediction = model(x)
```

- `eval()`: Layer의 동작 모드 변경
- `no_grad()`: Gradient 추적 중단


---

# 13. `ModuleList`

여러 Layer를 리스트처럼 관리하면서 PyTorch에 Module로 등록하려면 `nn.ModuleList`를 사용한다.


In [27]:
class ModuleListNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.layers = nn.ModuleList([nn.Linear(4, 8), nn.Linear(8, 8), nn.Linear(8, 2)])
        self.relu = nn.ReLU()

    def forward(self, x):
        for layer in self.layers[:-1]:
            x = self.relu(layer(x))

        return self.layers[-1](x)


model = ModuleListNetwork()

print(model)
print("Parameter 수:", sum(p.numel() for p in model.parameters()))

ModuleListNetwork(
  (layers): ModuleList(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): Linear(in_features=8, out_features=8, bias=True)
    (2): Linear(in_features=8, out_features=2, bias=True)
  )
  (relu): ReLU()
)
Parameter 수: 130


---

# 14. Forward Hook으로 중간 shape 확인


In [20]:
model = TwoLayerNetwork(4, 8, 3)


def print_shape_hook(module, inputs, output):
    print(
        f"{module.__class__.__name__}: "
        f"input={tuple(inputs[0].shape)}, "
        f"output={tuple(output.shape)}"
    )


handle = model.fc1.register_forward_hook(print_shape_hook)

x = torch.randn(5, 4)
output = model(x)

handle.remove()

Linear: input=(5, 4), output=(5, 8)


---

# 15. 연습문제

## 문제 1

다음 모델을 작성하자.

```text
입력 10
→ Linear(10, 16)
→ ReLU
→ Linear(16, 4)
```


In [28]:
class PracticeNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 4)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


model = PracticeNetwork()
x = torch.randn(8, 10)

output = model(x)

print(model)
print("출력 shape:", output.shape)

PracticeNetwork(
  (fc1): Linear(in_features=10, out_features=16, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=16, out_features=4, bias=True)
)
출력 shape: torch.Size([8, 4])


## 문제 2

문제 1 모델의 전체 Parameter 수를 계산하자.


In [29]:
total_parameters = sum(parameter.numel() for parameter in model.parameters())

print("전체 Parameter 수:", total_parameters)

전체 Parameter 수: 244


## 문제 3

문제 1과 같은 구조를 `nn.Sequential`로 작성하자.


In [30]:
sequential_model = nn.Sequential(nn.Linear(10, 16), nn.ReLU(), nn.Linear(16, 4))

print(sequential_model)

Sequential(
  (0): Linear(in_features=10, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=4, bias=True)
)


---

# 16. 연습문제 정답


In [24]:
# 문제 1
class PracticeNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(10, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 4)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        return self.fc2(x)


model = PracticeNetwork()
x = torch.randn(8, 10)

output = model(x)

print(model)
print("출력 shape:", output.shape)

PracticeNetwork(
  (fc1): Linear(in_features=10, out_features=16, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=16, out_features=4, bias=True)
)
출력 shape: torch.Size([8, 4])


In [25]:
# 문제 2
total_parameters = sum(parameter.numel() for parameter in model.parameters())

print("전체 Parameter 수:", total_parameters)

전체 Parameter 수: 244


계산:

$$
16\times10+16=176
$$

$$
4\times16+4=68
$$

$$
176+68=244
$$


In [26]:
# 문제 3
sequential_model = nn.Sequential(nn.Linear(10, 16), nn.ReLU(), nn.Linear(16, 4))

print(sequential_model)

Sequential(
  (0): Linear(in_features=10, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=4, bias=True)
)


---

# 17. 핵심 정리

- `nn.Module`: 모델과 Layer의 기본 클래스
- `__init__()`: Layer 정의
- `forward()`: 연산 흐름 정의
- 모델 호출은 `model(x)` 사용
- `nn.Linear`: 선형 변환
- `nn.ReLU`: 비선형 활성화
- `nn.Sequential`: 순차 구조 표현
- `parameters()`: 학습 가능한 Parameter 확인
- `state_dict()`: Weight와 Bias 저장
- `train()` / `eval()`: 학습·평가 모드 전환
- 모델과 입력은 같은 Device에 있어야 함
